数据去重

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# ===================== 1. 基础配置 =====================
# 读取数据
df = pd.read_excel('Aged forged Al alloy.xlsx')

# 特征列 + 目标列
feature_cols = ['Si','Fe','Cu','Mn','Mg','Cr','Zn','V','Ti','Zr','Li','Ni','Be','Sc','Tsol','Tage','tage']
target_cols = ['YS', 'UTS', 'El']

print(f'原始数据形状: {df.shape}')

# ===================== 2. 核心：带缺失值的智能去重 =====================
"""
去重规则（兼容目标值缺失）：
1. 完全重复行 → 直接删除
2. 特征完全相同，但目标值不同 → 保留标准差小的，大的删除
3. 特征相同 + 目标值有缺失 → 不参与标准差计算，直接合并取均值
"""

# 步骤1：删除完全重复的行
df = df.drop_duplicates().reset_index(drop=True)

# 步骤2：找出【特征完全重复】的组
dup_mask = df.duplicated(subset=feature_cols, keep=False)
dup_groups = df[dup_mask].groupby(feature_cols, dropna=False)

# 步骤3：删除目标值波动过大的组（std>30 视为异常）
drop_index = []
for _, group in dup_groups:
    # 只对非缺失值计算标准差
    target_std = group[target_cols].std(skipna=True).max()
    if target_std > 30:
        drop_index.extend(group.index.tolist())

df = df.drop(drop_index).reset_index(drop=True)

# 步骤4：剩余特征重复组 → 合并取均值（自动兼容缺失值）
final_dup_mask = df.duplicated(subset=feature_cols, keep=False)

# 非重复部分
df_unique = df[~final_dup_mask].copy()

# 重复部分合并（缺失值不影响取均值）
df_merged = df[final_dup_mask].groupby(feature_cols, dropna=False)[target_cols].mean().round(3).reset_index()

# 最终去重数据
df_final = pd.concat([df_unique, df_merged], ignore_index=True)

# ===================== 3. 输出结果 =====================
print(f'去重后数据形状: {df_final.shape}')
df_final.to_excel('dedup_and_merge.xlsx', index=False)
print('✅ 去重完成，已输出：dedup_and_merge.xlsx')

原始数据形状: (524, 23)
去重后数据形状: (491, 23)
✅ 去重完成，已输出：dedup_and_merge.xlsx
